In [1]:
import os
import json
import geopandas as gpd
import requests
import math
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import shutil # to delete if cloud level is more than 30%
import ForestFire_Opn_Tools as fireTools
import createNBRFiles_AndMaskwithSCL_eodmsCOG as createNBRFiles

In [2]:
# We could use a geojson or just a lat Long and 2km diameter search.
#Exported the sentinel2 footprints from the QGIS file as geojson
#geojson_path = './FireNearFlinFlonMay302025.geojson'
#geojson_path = './nwt_example.geojson'
#aoi_gdf = gpd.read_file(geojson_path)
#utm_crs = aoi_gdf.estimate_utm_crs().to_epsg()
#bbox = get_bbox(geojson_path)
"""
fireProjectName = "BC_FraserBostonBar"
lat = 49.9
#lon = -121.7
lon = -121.45 # had to move the longitude slightly to cover the full fire polygon
"""
fireProjectName = "Manitoba_SIndianLake"
lat = 56.65
lon = -99.5
bbox = fireTools.bbox_from_point(lat, lon, half_size_km=2)
bbox

[-99.53277481191363, 56.63198198198198, -99.46722518808637, 56.66801801801802]

In [3]:
# Filter assets by imaging date
start_date = '2026-06-27T00:00:00Z'
end_date = '2026-07-12T23:59:59Z'

In [4]:
# Open EODMS STAC catalog and explore available collections
ogc_api_url = 'https://www.eodms-sgdot.nrcan-rncan.gc.ca/search'
# Construct the collections endpoint
collections_url = f"{ogc_api_url}/collections"
# Send a GET request to the collections endpoint
response = requests.get(collections_url)
if response.status_code == 200:
    collections = response.json() 
    print("Collections available in the OGC API:")
    num_collections = len(collections.get("collections", []))
    print(f"Number of collections: {num_collections}")
    for collection in collections.get("collections", []):
        print(f"- {collection.get('id')}: {collection.get('title')}")
else:  
    print(f"Failed to retrieve collections. Status code: {response.status_code}")


Collections available in the OGC API:
Number of collections: 11
- Sentinel-1: Sentinel-1
- RCMImageProducts: RCMImageProducts
- rcm-ard: RADARSAT Constellation Mission, CEOS-ARD
- NAPL: NAPL
- Radarsat-1-Raw: Radarsat-1-Raw
- Radarsat-1-L1-COG: Radarsat-1-L1-COG
- Radarsat-1-FRED: Radarsat-1-FRED
- Sentinel-2: Sentinel-2
- r1-ard: RADARSAT-1, CEOS-ARD
- SGBAirPhotos: SGBAirPhotos
- Radarsat-2_Tropical_Forest_Products: Radarsat-2_Tropical_Forest_Products


In [5]:
# List all items in RCM-ARD collection that intersect with area of interest within date range
#collection_id = 'Sentinel2'
#collection_id = 'S2_L2A'
#collection_id = 'sentinel2'
collection_id = 'Sentinel-2'
datetime_range = f"{start_date}/{end_date}"

# Construct the search URL
search_url = f"{ogc_api_url}/collections/{collection_id}/items" #items.json for qgis
search_url

'https://www.eodms-sgdot.nrcan-rncan.gc.ca/search/collections/Sentinel-2/items'

In [6]:
# Set query parameters
# Observe that a limit has been set
params = {
    "datetime": datetime_range, #capital D for ms4w - Datetime
     "bbox":",".join(map(str, bbox)),
     "limit":100
}
params

{'datetime': '2026-06-27T00:00:00Z/2026-07-12T23:59:59Z',
 'bbox': '-99.53277481191363,56.63198198198198,-99.46722518808637,56.66801801801802',
 'limit': 100}

In [11]:
# Make the request
from datetime import datetime, timezone, timedelta
response = requests.get(search_url, params=params)
desired_properties = ["producttype", "acquisition_start", "product_link"]
now = datetime.now(timezone.utc)
filtered_links = []
bands = ["B12", "B11", "B09", "B8A", "B04", "SCL"]        
# Check and parse the response
if response.status_code == 200:
    cog_format = {}
    count = 0
    features = response.json().get("features", [])
    print(f"Found {len(features)} features between {start_date} and {end_date} including both level 1C and level 2A.")
    print("Wait until you see Download of all features completed to go to next step.")
    print("Download is saved if cloud level less than 30%, to create fire map, or its deleted")
    addFireProjectName_andProcessingDate = (f"./Process_Records/{fireProjectName}_%Y_%m_%d_processRecords.csv")
    resDataFileName= now.strftime(addFireProjectName_andProcessingDate)
    fireTools.start_a_new_csv_record(resDataFileName)
    print("Product Type","       ", "Acquisition start","          ", "Cloud Percent", "          ", "CRS", "          ", "Product Link", )
    for feature in features[:100]:      
        properties = feature.get("properties", {})
        product_type = properties.get('product:type')
        acquisition_start = properties.get('datetime')
        product_link = properties.get('product')
        ProcessDateTime = datetime.now()
        print(product_type)
        S2FileName = product_link.split("/Sentinel-2/")[1]
        if "MSIL2A" in product_link:
            # these asterisks help separate the log for each file processed
            print("**************************************************************************************************")
            count = count + 1
            inputDataDir, cloudPercent, crs = fireTools.process_product_link(product_link, fireProjectName)
            safeFileName_withSAFE_Ext = os.path.basename(inputDataDir)
            safeFileName = os.path.splitext(safeFileName_withSAFE_Ext)[0]
            print(product_type,"       ", acquisition_start,"          ", cloudPercent, "% cloud      ", crs, "          ", product_link)
            if cloudPercent > 50:
                Decision = "Dont create fire map"
                shutil.rmtree(inputDataDir, onerror=fireTools.remove_readonly)
            else:
                Decision = "Create fire map"
                filtered_links.append(inputDataDir)
                print(f"Starting to create COGS for bands in {inputDataDir}")
                cog_format[safeFileName] = fireTools.get_bands_as_cogs(bands, inputDataDir, fireProjectName) 
                shutil.rmtree(inputDataDir, onerror=fireTools.remove_readonly)
            table_data = [[ProcessDateTime, fireProjectName, S2FileName, cloudPercent, Decision]]
            fireTools.write_to_csv(resDataFileName, table_data)
    print("Download of all ", count, " level 2A features completed")    
    for product_link in filtered_links:
        print(product_link)
else:
    print(f"Failed to retrieve features. Status code: {response.status_code}")


Found 16 features between 2026-06-27T00:00:00Z and 2026-07-12T23:59:59Z including both level 1C and level 2A.
Wait until you see Download of all features completed to go to next step.
Download is saved if cloud level less than 30%, to create fire map, or its deleted
Product Type         Acquisition start            Cloud Percent            CRS            Product Link
S2MSI2A


AttributeError: 'NoneType' object has no attribute 'split'

In [ ]:
def plot_RGB_cog(rgb):
    # Normalize to [0, 1] for display
    rgb_normalized = rgb.astype(np.float32)
    rgb_normalized /= rgb_normalized.max()

    # Plot using matplotlib
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_normalized)
    plt.title("RGB Composite")
    plt.axis('off')
    plt.show()


In [ ]:
def plot_band_cog(band_cog_file):
    # Path to your COG file
    cog_path = band_cog_file

    # Open and read the first band
    with rasterio.open(cog_path) as src:
        band1 = src.read(1)

    # Plot the band
    plt.figure(figsize=(10, 8))
    plt.imshow(band1, cmap='gray')
    #aoi_gdf.to_crs(utm_crs).plot(ax=axes[0], facecolor='none', edgecolor='brown', linewidth=2)
    plt.colorbar(label='Pixel values')
    plt.title('Single Band Visualization of COG')
    plt.xlabel('Column Index')
    plt.ylabel('Row Index')
    plt.show()


In [ ]:
#PLEASE WAIT until it says "Completed storing all bands of all products as COG files" in the above step
for product, cog_list in cog_format.items():
    print(f"\nProduct: {product}")
    print(f"COG list: {cog_list}")

In [ ]:
standard_fire_composites = {}
standard_fire_composites_for_plots = {}
AshRich_fire_composites = {}
AshRich_fire_composites_for_plots = {}
output_dir_fireProj = "./Result_Images/FireComposite_Images/" + fireProjectName 
print(f"output directory is: {output_dir_fireProj}")
for product, cog_list in cog_format.items():
    output_dir = output_dir_fireProj + "/" + product
    print(f"Starting to create Fire Composites for {product}")
    print(f"If previous results are open in QGIS, close the project, otherwise it cant overwrite!")
    compositeType = "B12B8AB04_RGB"
    B12_cog = cog_list.get("B12")
    B8A_cog = cog_list.get("B8A")
    B04_cog = cog_list.get("B04")
    
    if not all([B12_cog, B8A_cog, B04_cog]):
        print(f"Missing bands for product {product}")

    rgb, rgb_forTif = fireTools.makeRGBComposite(B12_cog, B8A_cog, B04_cog)

    standard_fire_composites[product] = rgb_forTif
    standard_fire_composites_for_plots[product] = rgb
    Fire_RGB_COG = fireTools.save_to_GeoTif(rgb_forTif, compositeType, output_dir, B12_cog)
    print("Standard Fire Composite: Plotting the image for quick reference")
    plot_RGB_cog(rgb)
    print("SWIR - B12: Plotting the image for quick reference")
    plot_band_cog(B12_cog) # SWIR
    
    AshRich_compositeType = "B12B11B09_RGB"
    B11_cog = cog_list.get("B11")
    B09_cog = cog_list.get("B09")
    if not all([B12_cog, B11_cog, B09_cog]):
        print(f"Missing bands for product {product}")
    AshRich_rgb, AshRich_rgb_forTif = fireTools.makeRGBComposite(B12_cog, B11_cog, B09_cog)
    AshRich_fire_composites[product] = AshRich_rgb_forTif
    AshRich_fire_composites_for_plots[product] = AshRich_rgb
    AshRich_Fire_RGB_COG = fireTools.save_to_GeoTif(AshRich_rgb_forTif, AshRich_compositeType, output_dir, B12_cog)
    print("Ashlin Richardson Fire Composite: Plotting the image for quick reference")
    plot_RGB_cog(AshRich_rgb)
    #print("SWIR - B11: Plotting the image for quick reference")
    #plot_band_cog(B11) # SWIR
    #print("Water Vapour - B09: Plotting the image for quick reference")
    #plot_band_cog(B09) # water vapour
    #createNBRFiles.main(product)

In [ ]:
for product, cog_list in cog_format.items():
    createNBRFiles.main(fireProjectName, product)